# Ground Truth Analysis: Suspicious Event Labels

## Objective
Extract and analyze ALL labeled suspicious events from `suspicious.csv` files across all 12 cases to understand:
1. How many events are labeled as "Timestamp Manipulation" vs "Execution of Suspicious Programs"?
2. Which labeled events actually exist in the raw LogFile and UsnJrnl CSV files?
3. What is the actual ground truth dataset we're working with?

## Outputs
1. **`ground_truth_logfile_labeled.csv`** - All LogFile events with suspicious labels
2. **`ground_truth_usnjrnl_labeled.csv`** - All UsnJrnl events with suspicious labels
3. **Summary statistics and analysis tables**

---

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

BASE_DIR = Path('/Users/soni/Github/Digital-Detectives_Thesis')
RAW_DIR = BASE_DIR / 'data' / 'raw'
OUTPUT_DIR = BASE_DIR / 'data' / 'processed' / 'Phase 1 - Data Cleaning'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("✓ Libraries loaded")
print(f"✓ Base directory: {BASE_DIR}")
print(f"✓ Output directory: {OUTPUT_DIR}")

✓ Libraries loaded
✓ Base directory: /Users/soni/Github/Digital-Detectives_Thesis
✓ Output directory: /Users/soni/Github/Digital-Detectives_Thesis/data/processed/Phase 1 - Data Cleaning


---
## 2. Load and Match Suspicious Labels Across All Cases

In [2]:
print("=" * 80)
print("LOADING AND MATCHING SUSPICIOUS LABELS")
print("=" * 80)

all_labeled_logfile = []
all_labeled_usnjrnl = []
case_summary = []

for case_id in range(1, 13):
    print(f"\n{'=' * 80}")
    print(f"CASE {case_id}")
    print("=" * 80)
    
    # Load suspicious labels
    sus_file = RAW_DIR / 'suspicious' / f'{case_id:02d}-PE-Suspicious.csv'
    sus_df = pd.read_csv(sus_file, encoding='utf-8-sig')
    
    print(f"\n📂 Loaded suspicious labels: {len(sus_df)} total")
    
    # Count by category
    timestomp_labels = sus_df[sus_df['category'] == 'Timestamp Manipulation']
    tool_exec_labels = sus_df[sus_df['category'] == 'Execution of Suspicious Programs']
    
    print(f"  - Timestamp Manipulation: {len(timestomp_labels)}")
    print(f"  - Tool Execution: {len(tool_exec_labels)}")
    
    # Separate by source
    lf_labels = sus_df[sus_df['source'] == 'logfile']
    usn_labels = sus_df[sus_df['source'] == 'usnjrnl']
    
    print(f"\n📊 By source:")
    print(f"  - LogFile: {len(lf_labels)} labels")
    print(f"  - UsnJrnl: {len(usn_labels)} labels")
    
    # ==========================================
    # LOGFILE MATCHING
    # ==========================================
    if len(lf_labels) > 0:
        print(f"\n[LogFile] Loading raw data...")
        lf_file = RAW_DIR / 'logfile' / f'{case_id:02d}-PE-LogFile.csv'
        lf_df = pd.read_csv(lf_file, encoding='utf-8-sig', low_memory=False)
        print(f"  Raw LogFile records: {len(lf_df):,}")
        
        # Match by LSN
        lf_lsns = lf_labels['lsn/usn'].tolist()
        lf_matched = lf_df[lf_df['LSN'].isin(lf_lsns)].copy()
        
        print(f"  Labels to match: {len(lf_labels)}")
        print(f"  ✓ Matched events: {len(lf_matched)}")
        
        if len(lf_matched) < len(lf_labels):
            missing = len(lf_labels) - len(lf_matched)
            print(f"  ⚠️  Missing events: {missing} (labels exist but events not found in raw data)")
        
        # Add labels to matched events
        lf_matched['case_id'] = case_id
        
        # Merge with suspicious labels to get category and detail
        lf_matched = lf_matched.merge(
            lf_labels[['lsn/usn', 'category', 'detail']],
            left_on='LSN',
            right_on='lsn/usn',
            how='left'
        )
        
        # Rename for clarity
        lf_matched = lf_matched.rename(columns={
            'category': 'label_category',
            'detail': 'label_detail'
        })
        
        # Drop duplicate lsn/usn column
        lf_matched = lf_matched.drop(columns=['lsn/usn'], errors='ignore')
        
        all_labeled_logfile.append(lf_matched)
        
        # Show sample
        print(f"\n  Sample labeled LogFile events:")
        for idx, row in lf_matched.head(2).iterrows():
            print(f"    LSN {row['LSN']} | {row['label_category']} | {row['Event']}")
    
    # ==========================================
    # USNJRNL MATCHING
    # ==========================================
    if len(usn_labels) > 0:
        print(f"\n[UsnJrnl] Loading raw data...")
        usn_file = RAW_DIR / 'usnjrnl' / f'{case_id:02d}-PE-UsnJrnl.csv'
        usn_df = pd.read_csv(usn_file, encoding='utf-8-sig', low_memory=False)
        print(f"  Raw UsnJrnl records: {len(usn_df):,}")
        
        # Match by USN
        usn_usns = usn_labels['lsn/usn'].tolist()
        usn_matched = usn_df[usn_df['USN'].isin(usn_usns)].copy()
        
        print(f"  Labels to match: {len(usn_labels)}")
        print(f"  ✓ Matched events: {len(usn_matched)}")
        
        if len(usn_matched) < len(usn_labels):
            missing = len(usn_labels) - len(usn_matched)
            print(f"  ⚠️  Missing events: {missing} (labels exist but events not found in raw data)")
        
        # Add labels to matched events
        usn_matched['case_id'] = case_id
        
        # Merge with suspicious labels to get category and detail
        usn_matched = usn_matched.merge(
            usn_labels[['lsn/usn', 'category', 'detail']],
            left_on='USN',
            right_on='lsn/usn',
            how='left'
        )
        
        # Rename for clarity
        usn_matched = usn_matched.rename(columns={
            'category': 'label_category',
            'detail': 'label_detail'
        })
        
        # Drop duplicate lsn/usn column
        usn_matched = usn_matched.drop(columns=['lsn/usn'], errors='ignore')
        
        all_labeled_usnjrnl.append(usn_matched)
        
        # Show sample
        print(f"\n  Sample labeled UsnJrnl events:")
        for idx, row in usn_matched.head(2).iterrows():
            print(f"    USN {row['USN']} | {row['label_category']} | {row['EventInfo']}")
    
    # Store summary
    case_summary.append({
        'case_id': case_id,
        'total_labels': len(sus_df),
        'timestomp_labels': len(timestomp_labels),
        'tool_exec_labels': len(tool_exec_labels),
        'lf_labels': len(lf_labels),
        'lf_matched': len(lf_matched) if len(lf_labels) > 0 else 0,
        'lf_missing': len(lf_labels) - (len(lf_matched) if len(lf_labels) > 0 else 0),
        'usn_labels': len(usn_labels),
        'usn_matched': len(usn_matched) if len(usn_labels) > 0 else 0,
        'usn_missing': len(usn_labels) - (len(usn_matched) if len(usn_labels) > 0 else 0)
    })

print(f"\n\n{'=' * 80}")
print("✅ LOADING COMPLETE")
print("=" * 80)

LOADING AND MATCHING SUSPICIOUS LABELS

CASE 1

📂 Loaded suspicious labels: 4 total
  - Timestamp Manipulation: 2
  - Tool Execution: 2

📊 By source:
  - LogFile: 2 labels
  - UsnJrnl: 2 labels

[LogFile] Loading raw data...
  Raw LogFile records: 39,077
  Labels to match: 2
  ✓ Matched events: 2

  Sample labeled LogFile events:
    LSN 8729569062 | Execution of Suspicious Programs | File Creation
    LSN 8730038250 | Timestamp Manipulation | Time Reversal Event

[UsnJrnl] Loading raw data...
  Raw UsnJrnl records: 316,817
  Labels to match: 2
  ✓ Matched events: 2

  Sample labeled UsnJrnl events:
    USN 1327928416 | Execution of Suspicious Programs | File_Created / Data_Added / File_Closed
    USN 1328063200 | Timestamp Manipulation | Basic_Info_Changed / File_Closed

CASE 2

📂 Loaded suspicious labels: 3 total
  - Timestamp Manipulation: 1
  - Tool Execution: 2

📊 By source:
  - LogFile: 2 labels
  - UsnJrnl: 1 labels

[LogFile] Loading raw data...
  Raw LogFile records: 14,783
  

---
## 3. Combine All Labeled Events

In [3]:
print("\n" + "=" * 80)
print("COMBINING LABELED EVENTS")
print("=" * 80)

# Combine all LogFile events
if all_labeled_logfile:
    ground_truth_lf = pd.concat(all_labeled_logfile, ignore_index=True)
    print(f"\n✓ Total labeled LogFile events: {len(ground_truth_lf)}")
    print(f"  Across {ground_truth_lf['case_id'].nunique()} cases")
    
    # Category breakdown
    print(f"\n  By category:")
    for cat, count in ground_truth_lf['label_category'].value_counts().items():
        pct = count / len(ground_truth_lf) * 100
        print(f"    {count:>3} ({pct:>5.1f}%) | {cat}")
else:
    ground_truth_lf = pd.DataFrame()
    print("\n⚠️  No labeled LogFile events found")

# Combine all UsnJrnl events
if all_labeled_usnjrnl:
    ground_truth_usn = pd.concat(all_labeled_usnjrnl, ignore_index=True)
    print(f"\n✓ Total labeled UsnJrnl events: {len(ground_truth_usn)}")
    print(f"  Across {ground_truth_usn['case_id'].nunique()} cases")
    
    # Category breakdown
    print(f"\n  By category:")
    for cat, count in ground_truth_usn['label_category'].value_counts().items():
        pct = count / len(ground_truth_usn) * 100
        print(f"    {count:>3} ({pct:>5.1f}%) | {cat}")
else:
    ground_truth_usn = pd.DataFrame()
    print("\n⚠️  No labeled UsnJrnl events found")

print(f"\n{'=' * 80}")
print(f"GRAND TOTAL: {len(ground_truth_lf) + len(ground_truth_usn)} labeled events")
print("=" * 80)


COMBINING LABELED EVENTS

✓ Total labeled LogFile events: 22
  Across 11 cases

  By category:
     14 ( 63.6%) | Timestamp Manipulation
      8 ( 36.4%) | Execution of Suspicious Programs

✓ Total labeled UsnJrnl events: 246
  Across 11 cases

  By category:
    238 ( 96.7%) | Timestamp Manipulation
      8 (  3.3%) | Execution of Suspicious Programs

GRAND TOTAL: 268 labeled events


---
## 4. Summary Statistics

In [4]:
print("\n" + "=" * 80)
print("CASE-BY-CASE SUMMARY")
print("=" * 80)

summary_df = pd.DataFrame(case_summary)

print("\n")
print(summary_df.to_string(index=False))

print(f"\n\n{'=' * 80}")
print("TOTALS")
print("=" * 80)

print(f"\nLabels in suspicious.csv:")
print(f"  Total labels: {summary_df['total_labels'].sum()}")
print(f"  - Timestamp Manipulation: {summary_df['timestomp_labels'].sum()}")
print(f"  - Tool Execution: {summary_df['tool_exec_labels'].sum()}")

print(f"\nLogFile:")
print(f"  Labels: {summary_df['lf_labels'].sum()}")
print(f"  Matched: {summary_df['lf_matched'].sum()} ({summary_df['lf_matched'].sum()/summary_df['lf_labels'].sum()*100:.1f}%)")
print(f"  Missing: {summary_df['lf_missing'].sum()} ({summary_df['lf_missing'].sum()/summary_df['lf_labels'].sum()*100:.1f}%)")

print(f"\nUsnJrnl:")
print(f"  Labels: {summary_df['usn_labels'].sum()}")
print(f"  Matched: {summary_df['usn_matched'].sum()} ({summary_df['usn_matched'].sum()/summary_df['usn_labels'].sum()*100:.1f}%)")
print(f"  Missing: {summary_df['usn_missing'].sum()} ({summary_df['usn_missing'].sum()/summary_df['usn_labels'].sum()*100:.1f}%)")

print(f"\nGround Truth Dataset:")
total_matched = summary_df['lf_matched'].sum() + summary_df['usn_matched'].sum()
total_labels = summary_df['lf_labels'].sum() + summary_df['usn_labels'].sum()
print(f"  Total matched: {total_matched} / {total_labels} ({total_matched/total_labels*100:.1f}%)")
print(f"  Total missing: {total_labels - total_matched} ({(total_labels - total_matched)/total_labels*100:.1f}%)")

# Save summary
summary_file = OUTPUT_DIR / 'ground_truth_summary.csv'
summary_df.to_csv(summary_file, index=False)
print(f"\n✓ Summary saved: {summary_file.name}")


CASE-BY-CASE SUMMARY


 case_id  total_labels  timestomp_labels  tool_exec_labels  lf_labels  lf_matched  lf_missing  usn_labels  usn_matched  usn_missing
       1             4                 2                 2          2           2           0           2            2            0
       2             3                 1                 2          2           2           0           1            1            0
       3             4                 2                 2          2           2           0           2            2            0
       4            58                58                 0          1           1           0          57            1           56
       5             1                 1                 0          1           1           0           0            0            0
       6            72                72                 0          2           2           0          70           69            1
       7             4                 2            

---
## 5. Analyze LogFile Labeled Events

In [5]:
if len(ground_truth_lf) > 0:
    print("=" * 80)
    print("LOGFILE LABELED EVENTS ANALYSIS")
    print("=" * 80)
    
    print(f"\nTotal: {len(ground_truth_lf)} events")
    
    # Event type distribution
    print("\nEvent Type Distribution:")
    for event, count in ground_truth_lf['Event'].value_counts().items():
        pct = count / len(ground_truth_lf) * 100
        print(f"  {count:>3} ({pct:>5.1f}%) | {event}")
    
    # Redo operation distribution
    print("\nRedo Operation Distribution:")
    for redo, count in ground_truth_lf['Redo'].value_counts().items():
        pct = count / len(ground_truth_lf) * 100
        print(f"  {count:>3} ({pct:>5.1f}%) | {redo}")
    
    # Category breakdown
    print("\nBy Label Category:")
    for cat in ground_truth_lf['label_category'].unique():
        subset = ground_truth_lf[ground_truth_lf['label_category'] == cat]
        print(f"\n  {cat}: {len(subset)} events")
        print(f"    Cases: {sorted(subset['case_id'].unique().tolist())}")
        print(f"    Event types: {subset['Event'].unique().tolist()}")
    
    # Display sample rows
    print("\n" + "=" * 80)
    print("SAMPLE LOGFILE LABELED EVENTS (first 5)")
    print("=" * 80)
    display_cols = ['case_id', 'LSN', 'Event', 'label_category', 'File/Directory Name', 'Detail']
    print(ground_truth_lf[display_cols].head(5).to_string(index=False))
else:
    print("No LogFile labeled events to analyze")

LOGFILE LABELED EVENTS ANALYSIS

Total: 22 events

Event Type Distribution:
   14 ( 63.6%) | Time Reversal Event
    8 ( 36.4%) | File Creation

Redo Operation Distribution:
   14 ( 63.6%) | Update Resident Value
    8 ( 36.4%) | Initialize File Record Segment

By Label Category:

  Execution of Suspicious Programs: 8 events
    Cases: [1, 2, 3, 7, 8, 9, 11, 12]
    Event types: ['File Creation']

  Timestamp Manipulation: 14 events
    Cases: [1, 2, 3, 4, 5, 6, 7, 8, 9, 11, 12]
    Event types: ['Time Reversal Event']

SAMPLE LOGFILE LABELED EVENTS (first 5)
 case_id         LSN               Event                   label_category               File/Directory Name                                                                             Detail
       1  8729569062       File Creation Execution of Suspicious Programs   NEWFILETIME_X64.EXE-6C60D39A.pf                                                                                NaN
       1  8730038250 Time Reversal Event           T

---
## 6. Analyze UsnJrnl Labeled Events

In [6]:
if len(ground_truth_usn) > 0:
    print("=" * 80)
    print("USNJRNL LABELED EVENTS ANALYSIS")
    print("=" * 80)
    
    print(f"\nTotal: {len(ground_truth_usn)} events")
    
    # EventInfo distribution (top 10)
    print("\nEventInfo Distribution (top 10):")
    for event, count in ground_truth_usn['EventInfo'].value_counts().head(10).items():
        pct = count / len(ground_truth_usn) * 100
        print(f"  {count:>3} ({pct:>5.1f}%) | {event}")
    
    # Category breakdown
    print("\nBy Label Category:")
    for cat in ground_truth_usn['label_category'].unique():
        subset = ground_truth_usn[ground_truth_usn['label_category'] == cat]
        print(f"\n  {cat}: {len(subset)} events")
        print(f"    Cases: {sorted(subset['case_id'].unique().tolist())}")
        
        # Check for Basic_Info_Change pattern
        has_basic = subset['EventInfo'].str.contains('Basic_Info_Change', na=False, case=False)
        has_close = subset['EventInfo'].str.contains('Close', na=False, case=False)
        both_combined = has_basic & has_close
        
        print(f"    Pattern analysis:")
        print(f"      - Contains 'Basic_Info_Change': {has_basic.sum()} ({has_basic.sum()/len(subset)*100:.1f}%)")
        print(f"      - Contains 'Close': {has_close.sum()} ({has_close.sum()/len(subset)*100:.1f}%)")
        print(f"      - Both combined in one event: {both_combined.sum()} ({both_combined.sum()/len(subset)*100:.1f}%)")
    
    # Display sample rows
    print("\n" + "=" * 80)
    print("SAMPLE USNJRNL LABELED EVENTS (first 5)")
    print("=" * 80)
    display_cols = ['case_id', 'USN', 'EventInfo', 'label_category', 'File/Directory Name']
    print(ground_truth_usn[display_cols].head(5).to_string(index=False))
else:
    print("No UsnJrnl labeled events to analyze")

USNJRNL LABELED EVENTS ANALYSIS

Total: 246 events

EventInfo Distribution (top 10):
  229 ( 93.1%) | File_Created / Basic_Info_Changed / Data_Added / Data_Overwritten / File_Closed
    9 (  3.7%) | Basic_Info_Changed / File_Closed
    8 (  3.3%) | File_Created / Data_Added / File_Closed

By Label Category:

  Execution of Suspicious Programs: 8 events
    Cases: [1, 2, 3, 7, 8, 9, 11, 12]
    Pattern analysis:
      - Contains 'Basic_Info_Change': 0 (0.0%)
      - Contains 'Close': 8 (100.0%)
      - Both combined in one event: 0 (0.0%)

  Timestamp Manipulation: 238 events
    Cases: [1, 3, 4, 6, 7, 8, 9, 10, 11, 12]
    Pattern analysis:
      - Contains 'Basic_Info_Change': 238 (100.0%)
      - Contains 'Close': 238 (100.0%)
      - Both combined in one event: 238 (100.0%)

SAMPLE USNJRNL LABELED EVENTS (first 5)
 case_id        USN                               EventInfo                   label_category                 File/Directory Name
       1 1327928416 File_Created / Data_Ad

---
## 7. Save Ground Truth Datasets

In [7]:
print("\n" + "=" * 80)
print("SAVING GROUND TRUTH DATASETS")
print("=" * 80)

# Save LogFile ground truth
if len(ground_truth_lf) > 0:
    lf_output = OUTPUT_DIR / 'ground_truth_logfile_labeled.csv'
    ground_truth_lf.to_csv(lf_output, index=False, encoding='utf-8-sig')
    file_size = lf_output.stat().st_size / 1024
    print(f"\n✓ LogFile ground truth saved:")
    print(f"  File: {lf_output.name}")
    print(f"  Records: {len(ground_truth_lf)}")
    print(f"  Size: {file_size:.2f} KB")
    print(f"  Columns: {len(ground_truth_lf.columns)}")

# Save UsnJrnl ground truth
if len(ground_truth_usn) > 0:
    usn_output = OUTPUT_DIR / 'ground_truth_usnjrnl_labeled.csv'
    ground_truth_usn.to_csv(usn_output, index=False, encoding='utf-8-sig')
    file_size = usn_output.stat().st_size / 1024
    print(f"\n✓ UsnJrnl ground truth saved:")
    print(f"  File: {usn_output.name}")
    print(f"  Records: {len(ground_truth_usn)}")
    print(f"  Size: {file_size:.2f} KB")
    print(f"  Columns: {len(ground_truth_usn.columns)}")

print(f"\n{'=' * 80}")
print("✅ GROUND TRUTH ANALYSIS COMPLETE")
print("=" * 80)


SAVING GROUND TRUTH DATASETS

✓ LogFile ground truth saved:
  File: ground_truth_logfile_labeled.csv
  Records: 22
  Size: 10.38 KB
  Columns: 16

✓ UsnJrnl ground truth saved:
  File: ground_truth_usnjrnl_labeled.csv
  Records: 246
  Size: 121.62 KB
  Columns: 13

✅ GROUND TRUTH ANALYSIS COMPLETE


---
## 8. Key Findings Summary

### Questions Answered:

1. **How many labeled events exist in suspicious.csv?**
   - See totals in Section 4

2. **How many of those labels actually exist in the raw data?**
   - See "Matched" vs "Missing" breakdown in Section 4

3. **What patterns do timestomped events have?**
   - LogFile: See Event Type and Redo Operation distributions in Section 5
   - UsnJrnl: See EventInfo and pattern analysis in Section 6

### Next Steps:

Based on the findings above, we can now:
1. Update our Phase 1 filters to capture ALL events in the ground truth dataset
2. Verify that our smart union strategy correctly identifies all timestomped events
3. Proceed to Phase 2 with confidence in our labeled dataset

---